# Inspect reported article results
This notebook reads archived aggregate results. It does not train a model and does not treat reported values as results of a new run.

In [ ]:
from pathlib import Path
import sys, zipfile, subprocess
PROJECT_ROOT = Path('/content/DMAE_Colab_Repository')
if not PROJECT_ROOT.exists():
    local = Path.cwd()
    if (local / 'src' / 'amd_oct').exists():
        PROJECT_ROOT = local
    else:
        from google.colab import files
        print('Upload DMAE_Colab_Repository.zip (code package, not the image database).')
        uploaded = files.upload()
        package = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
        if package is None:
            raise ValueError('Select the code ZIP package.')
        destination = Path('/content').resolve()
        with zipfile.ZipFile(package) as archive:
            for member in archive.infolist():
                resolved = (destination / member.filename).resolve()
                if not resolved.is_relative_to(destination):
                    raise ValueError('Unsafe archive path.')
            archive.extractall(destination)
if not (PROJECT_ROOT / 'src' / 'amd_oct').is_dir():
    raise FileNotFoundError('Set PROJECT_ROOT to the extracted code-package directory.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Code root:', PROJECT_ROOT)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
reported = pd.read_csv(PROJECT_ROOT / 'results/reported/supplementary_results.csv')
assert len(reported) == 90
summary = reported.groupby(['extractor', 'classifier'])['accuracy_percent'].agg(['mean', 'std'])
display(summary)
principal = reported[reported['extractor'].isin(['Custom CNN', 'VGG16'])]
pivot = principal.groupby(['classifier', 'extractor'])['accuracy_percent'].mean().unstack()
pivot.plot.barh(figsize=(9, 6))
plt.xlabel('Reported internal validation accuracy (%)')
plt.tight_layout()
plt.show()
